### University of Virginia
### DS 7200: Distributed Computing
### Lab: Supervised Learning
### Last Updated: August 20, 2023

Sabine Segaloff

---

#### Instructions

This project has two parts:
- Part I: Classification - build and apply a logistic regression model on the Wisconsin Breast Cancer dataset.
- Part II: Regression - build and apply a linear regression model on the California Housing dataset.

**Total Possible Points: 10**

---

#### Part I: Classification (5 POINTS)

Here are the specifications and grading breakdown:

- the target variable is `diagnosis`
- use `f1`, `f2` as predictors (1 PT)
- split data into 60% training set, 40% test set 
- standardize the predictors (1 PT)
- use seed=314 whenever a seed is needed
- fit a Logistic Regression model with an intercept (1 PT)
- compute and show the area under the ROC curve for the test set (2 PTS)

In [12]:
from pyspark.sql import SparkSession

DATA_FILEPATH = 'wisc_breast_cancer_w_fields.csv'

spark = SparkSession \
    .builder \
    .appName("Wisc BRCA") \
    .getOrCreate()

#### Enter code and solution

In [13]:
# load the data into DF

# inferSchema=True has Spark infer types instead of defaulting all to string
# note that it also makes the read not fully lazy (to guess the types Spark has to scan the file right away - this data is small enough that the extra pass over it isn't an issue)
cancer_df = spark.read.csv(DATA_FILEPATH, header=True, inferSchema=True)

# check that f1 and f2 came in as double (numeric) and that diagnosis is string
# cancer_df.printSchema()

> Spark's LogisticRegression requires a numeric label (0.0/1.0); it won't accept a string column. diagnosis is stored as "M" (malignant) / "B" (benign), so I need to convert it.

> I'm choosing malignant = 1 on purpose: it's the event we care about detecting, and the evaluator treats 1.0 as the positive class (same idea as event_level = "second" in yardstick).

> Note to self: this is a fixed rule, not something learned from the data, so it's safe to do before the train/test split. No leakage (unlike the scaler, which learns means/SDs and must be fit on train only).

In [14]:
# check what values diagnosis actually takes before converting
# (otherwise(0.0) below would silently label any typo/missing value as benign)
cancer_df.groupBy("diagnosis").count().show()  # .show() is an action so Spark actually runs here

+---------+-----+
|diagnosis|count|
+---------+-----+
|        B|  357|
|        M|  212|
+---------+-----+



In [19]:
from pyspark.sql import functions as F  # F is the normal alias for Spark's column functions

# create numeric label column: malignant (M) = 1.0, benign (B) = 0.0
cancer_df = cancer_df.withColumn(           # withColumn returns a NEW DataFrame (Spark DFs are immutable), so reassign
    "label",                                # "label" is the default labelCol LogisticRegression looks for
    F.when(F.col("diagnosis") == "M", 1.0)  # this works like dplyr's if_else / case_when
     .otherwise(0.0)                        # everything else -> 0.0 (safe to do since groupBy above showed only M and B)
)
# lazy: nothing runs yet (this just adds a step to the plan)

# confirm the mapping across all rows
# each diagnosis should map to exactly one label value
cancer_df.groupBy("diagnosis", "label").count().show()

+---------+-----+-----+
|diagnosis|label|count|
+---------+-----+-----+
|        B|  0.0|  357|
|        M|  1.0|  212|
+---------+-----+-----+



Each diagnosis maps to exactly one label (M to 1.0 and B to 0.0), and the counts match the original 217/357 so my conversion is correct.

>Now to assemble predictors.

>Spark models only accept a single vector column for the predictors (unlike R's y` ~ f1 + f2` formula), so `VectorAssembler` packs `f1` and `f2` into one `features` column.

>`VectorAssembler` is a transformer, not an estimator: there's no `.fit()` because it doesn't learn anything from the data. That's also why it's safe to run before the train/test split (no leakage).

>**Note to self:** assembling before the split saves code, not computation. Because Spark is lazy, `train_df` and `test_df` will each replay the full plan (read → label → assemble → split) when an action runs, unless I `.cache()`.

In [ ]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(inputCols=["f1", "f2"], # inputCols takes a list of column names (note plural: inputCols)
                            outputCol="features") # "features" is the default featuresCol LogisticRegression looks for

cancer_vec_df = assembler.transform(cancer_df)
cancer_vec_df.select("f1", "f2", "features", "label").show(3, truncate=False) # action (spark runs here)

+-----+-----+-------------+-----+
|f1   |f2   |features     |label|
+-----+-----+-------------+-----+
|17.99|10.38|[17.99,10.38]|1.0  |
|20.57|17.77|[20.57,17.77]|1.0  |
|19.69|21.25|[19.69,21.25]|1.0  |
+-----+-----+-------------+-----+
only showing top 3 rows


#### Part II: Regression (5 POINTS)

In this project, you will work with the California Home Price dataset to train a regression model and predict median home prices. Here are the specifications and grading breakdown:

- Scale the response variable median_house_value, dividing by 100000 (1 PT)

- Split data into train set (80%), test set (20%) using seed=314 (1 PT)

- Add new predictor: `rooms_per_household`

- In the training set, select all of these features and standardize them: (1 PT)

feats = ["total_bedrooms", 
         "population", 
         "households", 
         "median_income", 
         "rooms_per_household"]

- Fit a linear regression model on the training set with these parameters:

  - maxIter=10
  - regParam=0.3
  - elasticNetParam=0.8  


- Compute the MSE on the test set (2 PTS)

In [1]:
import os
import pandas as pd

from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

/opt/conda/lib/python3.7/site-packages/pyspark/bin/load-spark-env.sh: line 68: ps: command not found
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


23/03/03 02:27:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
23/03/03 02:27:29 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [2]:
DATA_FILEPATH2 = 'cal_housing_data_preproc_w_header.txt'

#### Enter code and solution